# Cross-Model Linear Predictability (R²)

Tests whether one model's representations are **linearly predictable** from another's via ridge regression.

Unlike kernel alignment metrics (mutual_knn, CKA) that test distance structure similarity, R² measures what fraction of one model's representation variance is linearly predictable from another's. Two representations can have identical kernels yet low R² (complex rotations preserve distances but scramble axes), or different kernels yet high R² (one is a linear transform of the other).

Self-contained, runs entirely on CPU with small models.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from torchvision.models.feature_extraction import create_feature_extractor

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

from metrics import remove_outliers

device = "cpu"
torch.set_grad_enabled(False)
print(f"Device: {device}")

## Dataset

Load a small slice of the WIT dataset (64 samples) — enough for R² estimation while keeping CPU runtime fast.

In [ ]:
NUM_SAMPLES = 64

dataset = load_dataset("minhuh/prh", revision="wit_1024", split=f"train[:{NUM_SAMPLES}]")
print(f"Loaded {len(dataset)} samples")
print(f"Keys: {dataset.column_names}")
print(f"Example text: {str(dataset[0]['text'][0])[:100]}...")

## R² Computation

Cross-validated R² via ridge regression. Uses `StandardScaler` + `RidgeCV` in a pipeline with 5-fold CV.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


def cross_model_r2(X, Y, n_folds=5):
    """Cross-validated R\u00b2 for predicting Y from X via ridge regression."""
    X_np, Y_np = X.numpy(), Y.numpy()
    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 10)))
    scores = cross_val_score(model, X_np, Y_np, cv=n_folds, scoring="r2")
    return scores.mean()


# Sanity check 1: random features should have R\u00b2 ~ 0
rand_X = torch.randn(200, 128)
rand_Y = torch.randn(200, 64)
r2_random = cross_model_r2(rand_X, rand_Y)
print(f"Random features -> R\u00b2 = {r2_random:.3f} (expected ~0)")

# Sanity check 2: linear transform should have R\u00b2 ~ 1
W = torch.randn(128, 64)
rand_Y_linear = rand_X @ W + 0.01 * torch.randn(200, 64)
r2_linear = cross_model_r2(rand_X, rand_Y_linear)
print(f"Linear transform -> R\u00b2 = {r2_linear:.3f} (expected ~1)")

## Vision Models — Extract Features

Use timm ViT models of increasing size (all pretrained on ImageNet-21k).

In [ ]:
VISION_MODELS = [
    "vit_tiny_patch16_224.augreg_in21k",
    "vit_small_patch16_224.augreg_in21k",
    "vit_base_patch16_224.augreg_in21k",
]

vision_results = {}

for model_name in VISION_MODELS:
    print(f"\n--- {model_name} ---")
    model = timm.create_model(model_name, pretrained=True).eval()
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {num_params/1e6:.1f}M")

    # Build transform from model config
    transform = create_transform(
        **resolve_data_config(model.pretrained_cfg, model=model)
    )

    # Set up feature extraction at each transformer block
    num_blocks = len(model.blocks)
    return_nodes = [f"blocks.{i}.add_1" for i in range(num_blocks)]
    extractor = create_feature_extractor(model, return_nodes=return_nodes)

    # Extract features in batches
    batch_size = 16
    all_feats = []  # will be list of (batch, num_layers, feat_dim)

    for i in tqdm(range(0, NUM_SAMPLES, batch_size), desc="Extracting"):
        batch_end = min(i + batch_size, NUM_SAMPLES)
        ims = torch.stack([transform(dataset[j]["image"]) for j in range(i, batch_end)])
        output = extractor(ims)
        # CLS token (index 0) from each layer
        feats = torch.stack([v[:, 0, :] for v in output.values()]).permute(1, 0, 2)
        all_feats.append(feats)

    all_feats = torch.cat(all_feats, dim=0)  # (N, num_layers, feat_dim)
    print(f"Feature shape: {all_feats.shape}")

    vision_results[model_name] = {
        "feats": all_feats,
        "num_params": num_params,
        "modality": "vision",
    }
    del model, extractor

print("\nDone extracting vision features.")

## Language Models — Extract Features

Use a small Bloom model that fits comfortably on CPU.

In [ ]:
LANGUAGE_MODELS = [
    "bigscience/bloomz-560m",
]

texts = [str(x["text"][0]) for x in dataset]

language_results = {}

for model_name in LANGUAGE_MODELS:
    print(f"\n--- {model_name} ---")
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float32, output_hidden_states=True,
    ).eval()
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {num_params/1e6:.1f}M")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    tokens = tokenizer(texts, padding="longest", return_tensors="pt", truncation=True, max_length=128)

    batch_size = 8
    all_feats = []

    for i in tqdm(range(0, NUM_SAMPLES, batch_size), desc="Extracting"):
        batch_end = min(i + batch_size, NUM_SAMPLES)
        batch = {k: v[i:batch_end] for k, v in tokens.items()}
        output = model(**batch)

        # Average-pool over tokens (respecting attention mask)
        hidden_states = torch.stack(output.hidden_states).permute(1, 0, 2, 3)  # (B, L, T, D)
        mask = batch["attention_mask"].unsqueeze(1).unsqueeze(-1)  # (B, 1, T, 1)
        feats = (hidden_states * mask).sum(dim=2) / mask.sum(dim=2)  # (B, L, D)
        all_feats.append(feats)

    all_feats = torch.cat(all_feats, dim=0)
    print(f"Feature shape: {all_feats.shape}")

    language_results[model_name] = {
        "feats": all_feats,
        "num_params": num_params,
        "modality": "language",
    }
    del model, tokenizer

print("\nDone extracting language features.")

## Compute Cross-Modal R²

For each (LLM, ViT) pair, sweep all layer combinations and find the best R² (both directions).

In [ ]:
r2_results = []

for lang_name, lang_data in language_results.items():
    lang_feats = lang_data["feats"]  # (N, L_lang, D_lang)
    num_lang_layers = lang_feats.shape[1]

    for vis_name, vis_data in vision_results.items():
        vis_feats = vis_data["feats"]  # (N, L_vis, D_vis)
        num_vis_layers = vis_feats.shape[1]

        best_r2_fwd = -np.inf
        best_r2_rev = -np.inf
        best_r2_mean = -np.inf
        best_layers_fwd = None
        best_layers_rev = None
        best_layers_mean = None

        print(f"\n{lang_name} vs {vis_name} ({num_lang_layers} x {num_vis_layers} layer combos)")

        for lang_layer in tqdm(range(num_lang_layers), desc="Layer sweep"):
            lf = lang_feats[:, lang_layer, :]
            lf = remove_outliers(lf, q=0.95)
            lf = F.normalize(lf, p=2, dim=-1)

            for vis_layer in range(num_vis_layers):
                vf = vis_feats[:, vis_layer, :]
                vf = remove_outliers(vf, q=0.95)
                vf = F.normalize(vf, p=2, dim=-1)

                r2_fwd = cross_model_r2(lf, vf)  # lang -> vis
                r2_rev = cross_model_r2(vf, lf)  # vis -> lang
                r2_mean = (r2_fwd + r2_rev) / 2

                if r2_fwd > best_r2_fwd:
                    best_r2_fwd = r2_fwd
                    best_layers_fwd = (lang_layer, vis_layer)
                if r2_rev > best_r2_rev:
                    best_r2_rev = r2_rev
                    best_layers_rev = (lang_layer, vis_layer)
                if r2_mean > best_r2_mean:
                    best_r2_mean = r2_mean
                    best_layers_mean = (lang_layer, vis_layer)

        result = {
            "lang_model": lang_name,
            "vision_model": vis_name,
            "lang_params": lang_data["num_params"],
            "vision_params": vis_data["num_params"],
            "best_r2_fwd": best_r2_fwd,
            "best_r2_rev": best_r2_rev,
            "best_r2_mean": best_r2_mean,
            "best_layers_fwd": best_layers_fwd,
            "best_layers_rev": best_layers_rev,
            "best_layers_mean": best_layers_mean,
        }
        r2_results.append(result)

        print(f"  Best R\u00b2 fwd (lang->vis): {best_r2_fwd:.3f} at layers {best_layers_fwd}")
        print(f"  Best R\u00b2 rev (vis->lang): {best_r2_rev:.3f} at layers {best_layers_rev}")
        print(f"  Best R\u00b2 mean:            {best_r2_mean:.3f} at layers {best_layers_mean}")

## Plot Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot A: Best R\u00b2 vs vision model scale ---
ax = axes[0]
for r in r2_results:
    short_vis = r["vision_model"].split(".")[0].replace("vit_", "ViT-")
    ax.bar(
        short_vis,
        r["best_r2_mean"],
        color="steelblue",
        alpha=0.8,
        edgecolor="k",
        linewidth=0.5,
    )

ax.set_ylabel("Best Cross-Validated R\u00b2 (mean of fwd+rev)")
ax.set_title("Cross-Modal R\u00b2 vs Vision Model Scale\n(bloomz-560m \u2194 ViT)")
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(bottom=0)

# --- Plot B: R\u00b2 across language model layers (best vision layer fixed) ---
ax = axes[1]
colors_vis = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for idx, r in enumerate(r2_results):
    lang_feats = language_results[r["lang_model"]]["feats"]
    vis_feats = vision_results[r["vision_model"]]["feats"]
    best_vis_layer = r["best_layers_mean"][1]
    num_lang_layers = lang_feats.shape[1]

    r2_fwd_per_layer = []
    r2_rev_per_layer = []

    vf = vis_feats[:, best_vis_layer, :]
    vf = remove_outliers(vf, q=0.95)
    vf = F.normalize(vf, p=2, dim=-1)

    for lang_layer in range(num_lang_layers):
        lf = lang_feats[:, lang_layer, :]
        lf = remove_outliers(lf, q=0.95)
        lf = F.normalize(lf, p=2, dim=-1)
        r2_fwd_per_layer.append(cross_model_r2(lf, vf))
        r2_rev_per_layer.append(cross_model_r2(vf, lf))

    x = np.linspace(0, 1, num_lang_layers)
    short_vis = r["vision_model"].split(".")[0].replace("vit_", "ViT-")
    color = colors_vis[idx % len(colors_vis)]
    ax.plot(x, r2_fwd_per_layer, "-", color=color, label=f"{short_vis} (fwd)", alpha=0.8)
    ax.plot(x, r2_rev_per_layer, "--", color=color, label=f"{short_vis} (rev)", alpha=0.8)

ax.set_xlabel("Relative Language Model Layer Depth")
ax.set_ylabel("Cross-Validated R\u00b2")
ax.set_title("R\u00b2 Across Language Model Layers\n(vision layer fixed at best)")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("cross_model_r2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot to experiments/cross_model_r2.png")